# Pattern 6: Token Exchange (RFC 8693)

The MCP server receives the user's JWT, exchanges it with Keycloak for a narrowed token where the audience is scoped to just the target service. The azp (authorized party) identifies the agent for audit.

**What the service sees:** A cryptographically proven, audience-narrowed token. Least privilege.

**Weakness:** No fine-grained resource-level authorization. The user can access all resources allowed by role, but per-resource decisions are not enforced.

In [1]:
from framework.runner import PatternRunner
runner = PatternRunner("p06_token_exchange")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [2]:
runner.show_auth_code()

╭──────────────────────────────── p06_token_exchange/mcp_auth.py ────────────────────────────────╮
│    1 """Pattern 6: Token Exchange (RFC 8693).                                                  │
│    2                                                                                           │
│    3 The MCP server receives the user's JWT from the agent caller, exchanges it                │
│    4 with Keycloak for a narrowed token scoped to the target service, and                      │
│    5 forwards the exchanged token. This is the one pattern where the MCP server                │
│    6 transforms the credential rather than just forwarding it.                                 │
│    7 """                                                                                       │
│    8                                                                                           │
│    9 from framework.auth_helpers import exchange_token                                         │
│   10 from framework.config import TOOL_TO_TARGET_CLIENT                                        │
│   11 from framework.mcp.auth import AuthHandler                                                │
│   12                                                                                           │
│   13                                                                                           │
│   14 class TokenExchangeHandler(AuthHandler):                                                  │
│   15     def __init__(self):                                                                   │
│   16         self._current_tool_name: str | None = None                                        │
│   17                                                                                           │
│   18     async def before_tool_call(self, user_context, tool_name):                            │
│   19         self._current_tool_name = tool_name                                               │
│   20         return True                                                                       │
│   21                                                                                           │
│   22     async def prepare_request(self, user_context, headers):                               │
│   23         jwt = user_context.get("jwt")                                                     │
│   24         tool_name = self._current_tool_name or ""                                         │
│   25         if jwt:                                                                           │
│   26             target_audience = TOOL_TO_TARGET_CLIENT.get(tool_name)                        │
│   27             if target_audience is None:                                                   │
│   28                 raise ValueError(f"no target audience configured for tool {tool_name!r}") │
│   29             exchanged = exchange_token(jwt, target_audience)                              │
│   30             headers["Authorization"] = f"Bearer {exchanged}"                              │
│   31         return headers                                                                    │
│   32                                                                                           │
│   33                                                                                           │
│   34 auth_handler = TokenExchangeHandler()                                                     │
│   35                                                                                           │
╰────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────── p06_token_exchange/service_auth.py ───────────────────────────────────────╮
│    1 """Pattern 6: Token Exchange (service side).                                                               │
│    2                                                                                                            │
│    3 The JWT validation here is identical to pattern 5. The service validates                                   │
│    4 the signature via JWKS. The difference is what arrives: the exchanged token                                │
│    5 has aud scoped to just this service, so the service reports method="scoped_jwt".                           │
│    6                                                                                                            │
│    7 The magic happened at the MCP server (token exchange), not here.                                           │
│    8 """                                                                                                        │
│    9                                                                                                            │
│   10 import jwt as pyjwt                                                                                        │
│   11 from fastapi import Request                                                                                │
│   12 from jwt import PyJWKClient                                                                                │
│   13                                                                                                            │
│   14 from framework.config import EXPECTED_ISSUER, EXPENSE_SERVICE_CLIENT_ID, DOCUMENT_SERVICE_CLIENT_ID, JWKS_ │
│   15 from framework.services.identity import Identity                                                           │
│   16                                                                                                            │
│   17 _expense_jwk_client: PyJWKClient | None = None                                                             │
│   18 _document_jwk_client: PyJWKClient | None = None                                                            │
│   19                                                                                                            │
│   20                                                                                                            │
│   21 async def get_expense_identity(request: Request) -> Identity:                                              │
│   22     return await _validate_jwt(request, EXPENSE_SERVICE_CLIENT_ID, "_expense")                             │
│   23                                                                                                            │
│   24                                                                                                            │
│   25 async def get_document_identity(request: Request) -> Identity:                                             │
│   26     return await _validate_jwt(request, DOCUMENT_SERVICE_CLIENT_ID, "_document")                           │
│   27                                                                                                            │
│   28                                                                                                            │
│   29 async def _validate_jwt(request: Request, service_client_id: str, cache_key: str) -> Identity:             │
│   30     auth_header = request.headers.get("authorization")                                                     │
│   31     if not auth_header or not auth_header.lower().startswith("bearer "):                                   │
│   32         return Identity(method="none", detail="no auth provided"[38;2;248;248;2

The MCP server is the only pattern where it transforms the credential. It calls Keycloak's token exchange endpoint to get a new token where aud is narrowed to just the target service and azp identifies the agent.

This directly addresses the broad-audience risk from pattern 5: the exchanged token can only be used against the specific service it was minted for. If the expense service is compromised, the leaked token is useless against the document service.

In a production deployment, you would also request narrow scopes during the exchange (e.g. `expenses:read` only) to further restrict what the token can do. We focus on audience narrowing here since it demonstrates the key principle -- the mechanics of scope enforcement are the same (the service reads a token field and decides).

In [3]:
await runner.start()

[04/14/26 06:37:13] INFO     StreamableHTTP session manager started                  ]8;id=951516;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=294269;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=758236;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=293844;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:51051/mcp "HTTP/1.1 200 OK"        ]8;id=899525;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=395781;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=199783;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=344237;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=564059;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=787131;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:51051/mcp "HTTP/1.1 200 OK"        ]8;id=714140;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=280817;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:51051/mcp "HTTP/1.1 200 OK"        ]8;id=451414;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=108807;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:51052/mcp "HTTP/1.1 200 OK"        ]8;id=661133;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=623576;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

[04/14/26 06:38:06] INFO     StreamableHTTP session manager shutting down            ]8;id=715920;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=709039;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=283681;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=730670;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

## Run scenarios

Let's see this pattern in action with different users.

In [4]:
await runner.run_as("alice", "What are my recent expenses?")

[04/14/26 06:37:14] INFO     HTTP Request: POST                                                     ]8;id=875394;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=417102;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────╮
│ [alice] What are my recent expenses? │
╰──────────────────────────────────────╯

                    INFO     Terminating session: None                                       ]8;id=822538;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=627771;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=703868;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=51570;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=442192;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=988157;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

[04/14/26 06:37:19] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=539003;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=728373;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:37:28] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=58008;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=878313;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "scoped_jwt",                │
│     │              │      │        │ "identity_detail": "validated JWT issued by                                │
│     │              │      │        │ http://localhost:8080/realms/agentauth; aud=[\'expense-service-client\'];  │
│     │              │      │        │ azp=agent-client", "count": 4, "expenses": [{"id": 1, "user_id": "alice",  │
│     │              │      │        │ "department": "engineering", "amount": 42.5, "ca                           │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭─────────────────────────────────────────────────── answer ────────────────────────────────────────────────────╮
│ Here are your recent expenses (4 total):                                                                      │
│                                                                                                               │
│ - ID 1 | Department: Engineering | Software | $42.50 | JetBrains AI assistant subscription | Status: Approved │
│ - ID 2 | Department: Engineering | Travel | $156.00 | Train ticket to client offsite | Status: Approved       │
│ - ID 3 | Department: Engineering | Books | $89.00 | Designing Data-Intensive Applications | Status: Approved  │
│ - ID 4 | Department: Engineering | Hardware | $1,450.00 | External 4K monitor | Status: Pending               │
│                                                                                                               │
│ Totals:                                                                                                       │
│ - Approved: $287.50                                                                                           │
│ - Pending: $1,450.00                                                                                          │
│                                                                                                               │
│ Would you like me to view more details on any item or approve the pending one (ID 4) if you have permission?  │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your recent expenses (4 total):\n\n- ID 1 | Department: Engineering | Software | $42.50 | JetBrains AI assistant subscription | Status: Approved\n- ID 2 | Department: Engineering | Travel | $156.00 | Train ticket to client offsite | Status: Approved\n- ID 3 | Department: Engineering | Books | $89.00 | Designing Data-Intensive Applications | Status: Approved\n- ID 4 | Department: Engineering | Hardware | $1,450.00 | External 4K monitor | Status: Pending\n\nTotals:\n- Approved: $287.50\n- Pending: $1,450.00\n\nWould you like me to view more details on any item or approve the pending one (ID 4) if you have permission?', tool_calls=[ToolCallTrace(name='get_expenses', args={}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "scoped_jwt", "identity_detail": "validated JWT issued by http://localhost:8080/realms/agentauth; aud=[\\\'expense-service-client\\\']; azp=agent-client", "count": 4, "expenses": [{"id": 1, "user_id": "alice

In [5]:
await runner.run_as("bob", "Show me all expenses and approve expense 4")

                    INFO     HTTP Request: POST                                                     ]8;id=644952;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=345762;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────────────╮
│ [bob] Show me all expenses and approve expense 4 │
╰──────────────────────────────────────────────────╯

[04/14/26 06:37:30] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=168514;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=209470;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:37:40] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=445196;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=452070;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=476591;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=440302;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:37:50] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=843160;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=399895;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:37:54] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=88016;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=321315;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args                 ┃ status ┃ result                                                  ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses    │ {'department': None} │      - │ {'type': 'text', 'text': '{"identity_method":           │
│     │                 │                      │        │ "scoped_jwt", "identity_detail": "validated JWT issued  │
│     │                 │                      │        │ by http://localhost:8080/realms/agentauth;              │
│     │                 │                      │        │ aud=[\'expense-service-client\']; azp=agent-client",    │
│     │                 │                      │        │ "count": 6, "expenses": [{"id": 1, "user_id": "alice",  │
│     │                 │                      │        │ "department": "engineering", "amount": 42.5, "ca        │
│ 2   │ approve_expense │ {'expense_id': 4}    │      - │ {'type': 'text', 'text': '{"identity_method":           │
│     │                 │                      │        │ "scoped_jwt", "approved_by": "bob",                     │
│     │                 │                      │        │ "tool_side_authz_enabled": false, "expense": {"id": 4,  │
│     │                 │                      │        │ "user_id": "alice", "department": "engineering",        │
│     │                 │                      │        │ "amount": 1450.0, "category": "hardware",               │
│     │                 │                      │        │ "description": "External 4K monitor", "status":         │
│     │                 │                      │        │ "approved"}, "_status":                                 │
│ 3   │ get_expenses    │ {'department': None} │      - │ {'type': 'text', 'text': '{"identity_method":           │
│     │                 │                      │        │ "scoped_jwt", "identity_detail": "validated JWT issued  │
│     │                 │                      │        │ by http://localhost:8080/realms/agentauth;              │
│     │                 │                      │        │ aud=[\'expense-service-client\']; azp=agent-client",    │
│     │                 │                      │        │ "count": 6, "expenses": [{"id": 1, "user_id": "alice",  │
│     │                 │                      │        │ "department": "engineering", "amount": 42.5, "ca        │
└─────┴─────────────────┴──────────────────────┴────────┴─────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are all expenses (updated):                                                                                │
│                                                                                                                 │
│ - ID 1: user alice, department engineering, amount 42.50, category software, description JetBrains AI assistant │
│ subscription, status approved                                                                                   │
│ - ID 2: user alice, department engineering, amount 156.00, category travel, description Train ticket to client  │
│ offsite, status approved                                                                                        │
│ - ID 3: user alice, department engineering, amount 89.00, category books, description Designing Data-Intensive  │
│ Applications, status approved                                                                                   │
│ - ID 4: user alice, department engineering, amount 1450.00, category hardware, description External 4K monitor, │
│ status approved                                                                                                 │
│ - ID 5: user bob, department engineering, amount 320.00, category training, description OAuth 2.0 deep-dive     │
│ workshop, status approved                                                                                       │
│ - ID 6: user bob, department engineering, amount 67.00, category meals, description Team lunch after the        │
│ migration shipped, status approved                                                                              │
│                                                                                                                 │
│ Expense 4 has been approved (by bob). If you’d like, I can filter or export these, or approve/review another    │
│ expense.                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are all expenses (updated):\n\n- ID 1: user alice, department engineering, amount 42.50, category software, description JetBrains AI assistant subscription, status approved\n- ID 2: user alice, department engineering, amount 156.00, category travel, description Train ticket to client offsite, status approved\n- ID 3: user alice, department engineering, amount 89.00, category books, description Designing Data-Intensive Applications, status approved\n- ID 4: user alice, department engineering, amount 1450.00, category hardware, description External 4K monitor, status approved\n- ID 5: user bob, department engineering, amount 320.00, category training, description OAuth 2.0 deep-dive workshop, status approved\n- ID 6: user bob, department engineering, amount 67.00, category meals, description Team lunch after the migration shipped, status approved\n\nExpense 4 has been approved (by bob). If you’d like, I can filter or export these, or approve/review another expen

In [6]:
await runner.run_as("dave", "Search for compensation documents")

                    INFO     HTTP Request: POST                                                     ]8;id=1515;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=779522;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────╮
│ [dave] Search for compensation documents │
╰──────────────────────────────────────────╯

[04/14/26 06:37:56] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=796798;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=114718;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:37:57] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=902185;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=679861;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=295250;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=963206;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:38:01] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=829743;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=375947;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:38:05] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=324301;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=346866;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool             ┃ args                  ┃ status ┃ result                                                ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ search_documents │ {'q': 'compensation'} │      - │ {'type': 'text', 'text': '{"identity_method":         │
│     │                  │                       │        │ "scoped_jwt", "identity_detail": "validated JWT       │
│     │                  │                       │        │ issued by http://localhost:8080/realms/agentauth;     │
│     │                  │                       │        │ aud=[\'document-service-client\']; azp=agent-client", │
│     │                  │                       │        │ "allowed_groups": ["admin", "platform", "public"],    │
│     │                  │                       │        │ "count": 1, "documents": [{"id": 6, "title": "Compe   │
└─────┴──────────────────┴───────────────────────┴────────┴───────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Found 1 document matching "compensation":                                                                       │
│                                                                                                                 │
│ 1) ID 6 — "Compensation bands H1 2026"                                                                          │
│ - Snippet: "L3 75-95k, L4 95-130k, L5 130-180k, L6 180-240k. Confidential."                                     │
│ - Owner: dave                                                                                                   │
│ - Access: admin                                                                                                 │
│                                                                                                                 │
│ Would you like to view the full text of this document or search for additional compensation-related documents   │
│ (e.g., compensation policy, salary bands for other periods)?                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Found 1 document matching "compensation":\n\n1) ID 6 — "Compensation bands H1 2026"\n- Snippet: "L3 75-95k, L4 95-130k, L5 130-180k, L6 180-240k. Confidential."\n- Owner: dave\n- Access: admin\n\nWould you like to view the full text of this document or search for additional compensation-related documents (e.g., compensation policy, salary bands for other periods)?', tool_calls=[ToolCallTrace(name='search_documents', args={'q': 'compensation'}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "scoped_jwt", "identity_detail": "validated JWT issued by http://localhost:8080/realms/agentauth; aud=[\\\'document-service-client\\\']; azp=agent-client", "allowed_groups": ["admin", "platform", "public"], "count": 1, "documents": [{"id": 6, "title": "Compe', error=None)])

In [7]:
from framework.auth_helpers import fetch_user_jwt, exchange_token
from framework.display import compare_tokens
broad = fetch_user_jwt("alice")
exchanged = exchange_token(broad, "expense-service-client")
compare_tokens(broad, exchanged, label_a="broad JWT", label_b="exchanged JWT")

                    INFO     HTTP Request: POST                                                     ]8;id=881337;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=977024;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: POST                                                     ]8;id=853443;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=358066;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

                                           broad JWT  vs  exchanged JWT                                            
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ claim              ┃ broad JWT                                    ┃ exchanged JWT                               ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ sub                │ 7f78567c-4e16-44d4-b723-dca7328abd88         │ 7f78567c-4e16-44d4-b723-dca7328abd88        │
│ preferred_username │ alice                                        │ alice                                       │
│ aud                │ [document-service-client,                    │ expense-service-client                      │
│                    │ expense-service-client]                      │                                             │
│ azp                │ agent-client                                 │ agent-client                                │
│ iss                │ http://localhost:8080/realms/agentauth       │ http://localhost:8080/realms/agentauth      │
│ role               │ employee                                     │ employee                                    │
│ department         │ engineering                                  │ engineering                                 │
│ reports_to         │ bob                                          │ bob                                         │
│ scope              │ openid agent-claims                          │ openid agent-claims                         │
│ exp                │ 1776175685                                   │ 1776175685                                  │
│ iat                │ 1776173885                                   │ 1776173885                                  │
│ jti                │ onrtro:ac29fbec-0d53-f773-1b31-c8556e3aa311  │ onrtte:2277ce2b-4baf-1d46-28f9-d1f9a65c872a │
│ sid                │ wJDxMdGEqIDK2Z54zA8ZHbre                     │ wJDxMdGEqIDK2Z54zA8ZHbre                    │
│ typ                │ Bearer                                       │ Bearer                                      │
└────────────────────┴──────────────────────────────────────────────┴─────────────────────────────────────────────┘

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [8]:
await runner.show_service_identity()

                    INFO     HTTP Request: GET http://127.0.0.1:51049/debug/last-request "HTTP/1.1  ]8;id=291161;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=465188;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭────────────────────────────────────── expense-service /debug/last-request ──────────────────────────────────────╮
│ method:  scoped_jwt                                                                                             │
│ user_id: bob                                                                                                    │
│ detail:  validated JWT issued by http://localhost:8080/realms/agentauth; aud=['expense-service-client'];        │
│ azp=agent-client                                                                                                │
│                                                                                                                 │
│ claims:  sub=6e12d041..., role=manager, department=engineering, aud=expense-service-client, azp=agent-client    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:51050/debug/last-request "HTTP/1.1  ]8;id=445257;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=487872;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭───────────────────────────────────── document-service /debug/last-request ──────────────────────────────────────╮
│ method:  scoped_jwt                                                                                             │
│ user_id: dave                                                                                                   │
│ detail:  validated JWT issued by http://localhost:8080/realms/agentauth; aud=['document-service-client'];       │
│ azp=agent-client                                                                                                │
│                                                                                                                 │
│ claims:  sub=bc99f071..., role=admin, department=platform, aud=document-service-client, azp=agent-client        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The service reports method=scoped_jwt. The audience is narrowed to just this service's client ID. If this token leaks, it only works against the expense service.

The azp=agent-client provides an audit trail: the service knows the agent initiated this request on behalf of the user.

### A note on scopes vs. claims

This repo uses custom JWT claims (role, department, reports_to) for authorization decisions. OAuth2 also supports **token scopes** -- strings like `expenses:read` or `expenses:approve` that declare what operations a token is allowed to perform.

From the service's perspective, the mechanics are the same: read a field from the token, enforce it in code. Claims describe *who the user is* (role, department). Scopes describe *what this token is allowed to do*. Both require the service to check them -- neither is enforced automatically.

Token exchange is where scopes become especially relevant in production: you could request a narrow scope during the exchange (e.g. `expenses:read` only) so the exchanged token can't approve expenses even if the user's role would allow it. We don't implement scope-based restrictions here to keep the focus on the identity progression, but in a production deployment you'd likely combine audience scoping (what we show) with scope restrictions.

### What's still missing

We still lack per-resource authorization. Any manager can approve any expense in their department. We need fine-grained access control.

In [9]:
await runner.stop()

Pattern p06_token_exchange stopped.

[04/14/26 06:38:07] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=946841;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=747965;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       